In [1]:
from pathlib import Path

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

processed_data_path = (
    project_root
    / "data"
    / "processed"
    / "heart_disease_clean.csv"
)

if not processed_data_path.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found: {processed_data_path}"
    )

df = pd.read_csv(processed_data_path)

target_column = "heart_disease"

if target_column not in df.columns:
    raise KeyError(
        f"Target column '{target_column}' was not found"
    )

if df[target_column].isna().any():
    raise ValueError(
        "The target column contains missing values"
    )

X = df.drop(columns=[target_column])
y = df[target_column].astype("int64")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Dataset loaded successfully")
print("Full dataset shape:", df.shape)
print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("Training positive rate:", round(y_train.mean() * 100, 2), "%")
print("Testing positive rate:", round(y_test.mean() * 100, 2), "%")

Dataset loaded successfully
Full dataset shape: (4238, 16)
Training feature shape: (3390, 15)
Testing feature shape: (848, 15)
Training positive rate: 15.19 %
Testing positive rate: 15.21 %


In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

continuous_features = [
    "age",
    "cigarettes_per_day",
    "total_cholesterol",
    "systolic_bp",
    "diastolic_bp",
    "bmi",
    "heart_rate",
    "glucose"
]

binary_features = [
    "current_smoker",
    "bp_meds",
    "prevalent_stroke",
    "prevalent_hypertension",
    "diabetes"
]

categorical_features = [
    "gender",
    "education"
]

all_defined_features = (
    continuous_features
    + binary_features
    + categorical_features
)

missing_features = sorted(
    set(X_train.columns) - set(all_defined_features)
)

unexpected_features = sorted(
    set(all_defined_features) - set(X_train.columns)
)

duplicate_features = [
    feature
    for feature in set(all_defined_features)
    if all_defined_features.count(feature) > 1
]

if missing_features:
    raise ValueError(
        f"Features not assigned to a group: {missing_features}"
    )

if unexpected_features:
    raise ValueError(
        f"Defined features not found: {unexpected_features}"
    )

if duplicate_features:
    raise ValueError(
        f"Features defined more than once: {duplicate_features}"
    )

logistic_continuous_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

tree_continuous_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

binary_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

logistic_preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous",
            logistic_continuous_pipeline,
            continuous_features
        ),
        (
            "binary",
            clone(binary_pipeline),
            binary_features
        ),
        (
            "categorical",
            clone(categorical_pipeline),
            categorical_features
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous",
            tree_continuous_pipeline,
            continuous_features
        ),
        (
            "binary",
            clone(binary_pipeline),
            binary_features
        ),
        (
            "categorical",
            clone(categorical_pipeline),
            categorical_features
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print("Preprocessing pipelines created successfully")
print("Continuous features:", len(continuous_features))
print("Binary features:", len(binary_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(all_defined_features))